# KEROSENE CMA-ES Baseline

RL과 같은 Aspen 조건과 reward 기준을 유지하면서, CMA-ES는 전용 evaluator 구조로 실행합니다.

In [ ]:
import importlib
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

module_directory = Path.cwd()
if str(module_directory) not in sys.path:
    sys.path.insert(0, str(module_directory))

import kerosene_cmaes_runner
importlib.reload(kerosene_cmaes_runner)
print('kerosene_cmaes_runner path:', kerosene_cmaes_runner.__file__)


In [ ]:
current_directory = Path.cwd()
aspen_filename = 'FT-PFR-3.27.apw'

SIM_VISIBLE = True
SIM_SUPPRESS_DIALOGS = False


def create_simulation():
    return kerosene_cmaes_runner.create_simulation(
        aspen_filename=aspen_filename,
        working_directory=str(current_directory),
        visible=SIM_VISIBLE,
        suppress_dialogs=SIM_SUPPRESS_DIALOGS,
    )


sim = create_simulation()
try:
    ok = sim.Run()
except Exception as err:
    ok = False
    print(f'Initial Aspen error: {err}')
print('Run Done, Converge:', ok)


In [ ]:
reward_report_scale = 10.0
action_low = np.array([220.0, 1.50, 0.30], dtype=np.float32)
action_high = np.array([260.0, 2.20, 0.85], dtype=np.float32)

# RL comparison conditions: same bounds, same reward formula, same 1-step static evaluation.
evaluator = kerosene_cmaes_runner.KeroseneCMAESEvaluator(
    simulation=sim,
    simulation_factory=create_simulation,
    action_low=action_low,
    action_high=action_high,
    reward_report_scale=reward_report_scale,
    dialog_suppression=SIM_SUPPRESS_DIALOGS,
)

max_evaluations = 200
validation_evaluations = 8
population_size = 8
sigma_init = 0.18

expt_code = '_expt401_static_cma_es'
run = 10
run_seed = {1: 212, 2: 123, 3: 456, 4: 9, 5: 11, 6: 88, 7: 101, 8: 1122, 9: 2002, 10: 5555}
seed = run_seed[run]

random.seed(seed)
np.random.seed(seed)

state, info = kerosene_cmaes_runner.validate_aspen_session(evaluator)
print('Preflight state:', state.tolist())


In [ ]:
def do_the_thing(experiment_max_num):
    print(f'[CMA-ES] do_the_thing start | experiment_max_num={experiment_max_num}')
    kerosene_cmaes_runner.validate_aspen_session(evaluator)
    results = []
    for num in range(experiment_max_num):
        filename = f'{expt_code}_{max_evaluations}eval_RNG{seed}_{run}_{num}'
        rewards = kerosene_cmaes_runner.run_cma_es_optimization(
            evaluator=evaluator,
            max_evaluations=max_evaluations,
            validation_evaluations=validation_evaluations,
            population_size=population_size,
            sigma_init=sigma_init,
            filename=filename,
            seed=seed,
        )
        results.append(rewards)
    print('Done with CMA-ES runs.')
    return results


experiment_max_num = 1
training_results = do_the_thing(experiment_max_num)


In [ ]:
report_dir = Path.cwd()
comparison_tags = [
    '_expt207_static_probe_anneal_sac',
    '_expt301_static_probe_anneal_td3',
    '_expt401_static_cma_es',
]


def infer_algorithm(file_name):
    lower_name = file_name.lower()
    if 'cma_es' in lower_name or 'cma-es' in lower_name or 'cma' in lower_name:
        return 'CMA-ES'
    if 'td3' in lower_name:
        return 'TD3'
    if 'sac' in lower_name:
        return 'SAC'
    return 'Unknown'


reward_csvs = sorted(
    path for path in report_dir.glob('report_rewards_*.csv')
    if any(tag in path.name for tag in comparison_tags)
)

print(f'Looking for comparison reward CSV files in: {report_dir}')

if reward_csvs:
    reward_df = pd.concat([pd.read_csv(csv_path).assign(file=csv_path.name) for csv_path in reward_csvs], ignore_index=True)
    if 'total_reward_display' in reward_df.columns:
        reward_col = 'total_reward_display'
        reward_title = 'Per-Episode Reward Comparison (Display Scale)'
    elif 'total_reward_raw' in reward_df.columns:
        reward_col = 'total_reward_raw'
        reward_title = 'Per-Episode Reward Comparison (Raw Reward)'
    else:
        reward_col = 'total_reward'
        reward_title = 'Per-Episode Reward Comparison'

    reward_df['algorithm'] = reward_df['file'].map(infer_algorithm)
    reward_df['best_so_far'] = reward_df.groupby('file')[reward_col].cummax()

    summary_df = reward_df.groupby(['algorithm', 'file'], as_index=False).agg(
        final_reward=(reward_col, 'last'),
        best_reward=('best_so_far', 'max'),
    )
    display(summary_df.sort_values(['best_reward', 'final_reward'], ascending=[False, False]))

    fig_best = px.line(
        reward_df,
        x='episode',
        y='best_so_far',
        color='algorithm',
        line_dash='file',
        title='Best-So-Far Reward Comparison',
    )
    fig_best.show()

    fig_reward = px.line(
        reward_df,
        x='episode',
        y=reward_col,
        color='algorithm',
        line_dash='file',
        title=reward_title,
    )
    fig_reward.show()
else:
    print('No comparison reward CSV files were found in the current directory.')


In [ ]:
try:
    sim.CloseAspen()
except Exception as exc:
    print(f'Aspen close skipped: {exc}')
